# SET Market Holiday Service — Official Market Closure Calendar

The authoritative list of days the Stock Exchange of Thailand is closed, in English or Thai.

**You will learn how to:**
- Fetch the official holiday calendar for the current year
- Ask whether a given day is a market holiday
- Find the next holiday and spot long weekends
- Read Thai holiday names, including SET's `*` footnote marker
- Handle this endpoint's unusually flaky HTTP 401 behaviour

> ⚠️ **Four API gotchas** (live-verified 2026-07-27):
> 1. **Only the current year is served.** 2024, 2025, 2027 and 2028 all return HTTP 401 while the current year returns 200. No history, no next-year lookahead.
> 2. **HTTP 401 is the only failure code** — and it is also returned *transiently* on valid requests. The service retries automatically.
> 3. `is_holiday()` means *"on SET's published holiday list"*, **not** *"the market is closed"* — weekends are not in the payload.
> 4. Descriptions are **verbatim**: a trailing `" *"` is a SET footnote and is never stripped.

## 1. Setup

In [ ]:
from datetime import date, datetime
from zoneinfo import ZoneInfo

from settfex.services.set import HolidayService, get_holidays
from settfex.utils.data_fetcher import FetcherConfig

BANGKOK = ZoneInfo("Asia/Bangkok")

# In Jupyter, await works directly (no asyncio.run needed)

## 2. Fetch this year's holiday calendar

With no arguments, `get_holidays()` resolves the year from the **Asia/Bangkok** clock — never your machine's local time.

In [ ]:
calendar = await get_holidays()

print(f"{calendar.count} market holidays in {calendar.year} (lang={calendar.lang})\n")
for holiday in calendar.holidays:
    print(f"{holiday.holiday_date:%Y-%m-%d %a}  {holiday.description}")

## 3. Is a given day a market holiday?

`is_holiday()` accepts a `date` or a `datetime`. Naive datetimes are treated as Bangkok-local; aware ones are converted to the Bangkok calendar day first.

In [ ]:
new_year = date(calendar.year, 1, 1)

print(f"{new_year} is a holiday? {calendar.is_holiday(new_year)}")
print(f"Details: {calendar.get_holiday(new_year)}")

# get_holiday() returns None for an ordinary trading day
print(f"\n{date(calendar.year, 1, 5)} -> {calendar.get_holiday(date(calendar.year, 1, 5))}")

## 4. ⚠️ Weekends are NOT in the payload

This is the single easiest way to get a wrong answer. The API publishes **holidays only**, so a Saturday is not a "holiday" even though the market is shut. Combine both checks yourself.

In [ ]:
def market_is_closed(day: date) -> bool:
    """Holidays alone are not enough — weekends must be handled separately."""
    return day.weekday() >= 5 or calendar.is_holiday(day)


for day in [date(calendar.year, 1, 1), date(calendar.year, 1, 3), date(calendar.year, 1, 5)]:
    print(
        f"{day:%Y-%m-%d %a}  is_holiday={calendar.is_holiday(day)!s:5s} "
        f"market_is_closed={market_is_closed(day)}"
    )

## 5. The next market holiday

`next_holiday()` is exclusive of the given day and defaults to today in Bangkok. It returns `None` once the year runs out — the calendar only covers its own year.

In [ ]:
today = datetime.now(BANGKOK).date()
upcoming = calendar.next_holiday()

if upcoming:
    days_away = (upcoming.holiday_date.date() - today).days
    print(
        f"Next holiday: {upcoming.holiday_date:%d %b %Y} ({days_away} days) — {upcoming.description}"
    )
else:
    print(f"No holidays left in {calendar.year}")

## 6. Filter by month — finding long weekends

In [ ]:
for month in range(1, 13):
    holidays = calendar.filter_by_month(month)
    if holidays:
        names = ", ".join(h.description for h in holidays)
        print(f"{month:2d}: {len(holidays)} — {names}")

## 7. Thai descriptions

The English and Thai payloads are 1:1 aligned — same dates, same order. Note that Thai parenthetical notes use Buddhist-era years (2569 = 2026).

In [ ]:
thai = await get_holidays(calendar.year, lang="th")

for en, th in zip(calendar.holidays, thai.holidays):
    print(f"{en.holiday_date:%Y-%m-%d}  {en.description[:42]:45s} {th.description}")

## 8. The verbatim `*` footnote marker

SET marks *additional special closures* with a trailing `" *"`. It is part of the published name and is deliberately never stripped — unlike other SET models, `Holiday` does not enable `str_strip_whitespace`.

In [ ]:
marked = [h for h in calendar.holidays if h.description.endswith(" *")]

print(f"{len(marked)} description(s) carry the footnote marker:")
for holiday in marked:
    print(f"  {holiday.holiday_date:%Y-%m-%d}  {holiday.description!r}")

## 9. Raw tier — the untouched API response

`fetch_holidays_raw()` is the escape hatch: a bare list of dicts exactly as the API returned them.

In [ ]:
service = HolidayService()
raw = await service.fetch_holidays_raw(calendar.year)

print(f"{len(raw)} entries, keys={set(raw[0])}")
raw[:3]

## 10. Only the current year is served

Requesting any other year returns HTTP 401 — the same code the endpoint uses for transient failures, so the error message spells out both causes.

In [ ]:
from settfex.exceptions import FetchError

try:
    await get_holidays(calendar.year - 1, config=FetcherConfig(max_retries=1))
except FetchError as exc:
    print(f"HTTP {exc.status_code}\n{exc}")

## 11. Handling the transient 401

This endpoint degrades the harder you poll it (~100% success cold, ~12% after ~150 requests) and recovers on its own when left alone. `HolidayService` retries `401`/`403`/`429` with exponential backoff, tuned through the normal `FetcherConfig` knobs.

In [ ]:
# 7 attempts, backing off 2s, 4s, 8s, 16s, 32s, 64s
patient = FetcherConfig(max_retries=6, retry_delay=2.0)
calendar = await get_holidays(config=patient)

print(f"Fetched {calendar.count} holidays for {calendar.year}")

# Holiday data is static for a whole year — cache it rather than re-fetching.

## 12. Export to pandas (optional: `pip install "settfex[dataframe]"`)

In [ ]:
try:
    import pandas as pd

    df = pd.DataFrame(
        {
            "date": [h.holiday_date.date() for h in calendar.holidays],
            "weekday": [h.holiday_date.strftime("%a") for h in calendar.holidays],
            "description": [h.description for h in calendar.holidays],
        }
    )
    display(df)
except ImportError:
    print("pandas not installed — pip install 'settfex[dataframe]'")

## Use Cases

- **Skip closed days** when iterating a date range for price or news queries
- **Validate a "latest trading day"** result against the published calendar
- **Plan around long weekends** (Songkran in April, the December cluster)
- **Surface upcoming closures** in a dashboard or trading UI

> ⚠️ Two limits to design around: this endpoint serves **only the current year**, and it lists
> **whole-day closures only** — it has no field for partial sessions or altered trading hours.
> Combined with the missing weekends, that means it is not by itself a trading calendar.

**See also**: `docs/settfex/services/set/holiday.md` · News (16) for disclosures · Chart Quotation (13) for the latest traded price · Latest Historical Trading (14) for the last trading day's summary